In [1]:
import ctypes
ctypes.CDLL(
    "/mnt/nvme_storage/shared/envs/data-science/lib/python3.11/site-packages/nvidia/cu13/lib/libnvJitLink.so.13",
    mode=ctypes.RTLD_GLOBAL,
)

<CDLL '/mnt/nvme_storage/shared/envs/data-science/lib/python3.11/site-packages/nvidia/cu13/lib/libnvJitLink.so.13', handle 558a8679c530 at 0x734d6c2d4e90>

In [2]:
import os
import json
import torch
import pandas as pd
from unsloth import FastModel
from datasets import Dataset

# ==========================================
# CONFIG
# ==========================================
MODEL_NAME = "google/gemma-4-E4B-it"
MAX_SEQ_LENGTH = 1024

# Path to the Chinese dev set (with gold labels) used for training.
# Update to wherever your organizer-provided dev file actually lives.
DEV_DATA_PATH = "/mnt/nvme_storage/shared/users/amasha/PlurVa/simplified_set/indonesian_simplified.json"

ADAPTER_DIR = "/mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/indonesian_adapter"
OUTPUT_ADAPTER_PATH = f"{ADAPTER_DIR}/indonesian_adapter"

# Small-VRAM-friendly settings (12-16GB target)
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8   # effective batch size 16
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4

# ==========================================
# PROMPT TEMPLATE
# (identical wording to prompt_chinese() in the inference script —
#  do not edit without updating inference to match)
# ==========================================
def get_option_text(row, letter):
    # Flat schema: Option_A / Option_B / Option_C / Option_D (also tries
    # lowercase/other casings some dumps use)
    for key in [f"Option_{letter.upper()}", f"option_{letter.lower()}",
                f"Option_{letter.lower()}", f"option_{letter.upper()}"]:
        if key in row and pd.notna(row.get(key)) and str(row.get(key)).strip() != "":
            return row[key]
    # Nested schema: options = {"A": ..., "B": ..., ...}
    options = row.get("options")
    if isinstance(options, dict):
        val = options.get(letter.upper(), options.get(letter.lower(), ""))
        if val:
            return val
    raise KeyError(f"Could not find option '{letter}' in row (id={get_row_id(row)}).")

def get_row_id(row, idx=None):
    for key in ["id", "ID", "Id", "question_id"]:
        if key in row and pd.notna(row.get(key)):
            return row[key]
    return idx

def get_question_text(row):
    for key in ["scenario_question", "Question", "question", "Scenario"]:
        val = row.get(key)
        if val is not None and str(val).strip() != "":
            return val
    raise KeyError(f"Could not find question text in row (id={row.get('id')}).")

def prompt_chinese(row):
    opt_a, opt_b = get_option_text(row, 'A'), get_option_text(row, 'B')
    opt_c, opt_d = get_option_text(row, 'C'), get_option_text(row, 'D')
    return (
        f"<start_of_turn>user\n"
        f"You are a Indonesian Expert for answering multiple-choice questions. \n"
        f"    You need to always evaluate these questions based on the Indonesian context and provide the most accurate answer.\n"
        f"    You should only respond with the correct option text, without any additional explanation or commentary.\n"
        f"Question: {get_question_text(row)}\n"
        f"Option A: {opt_a}\n"
        f"Option B: {opt_b}\n"
        f"Option C: {opt_c}\n"
        f"Option D: {opt_d}\n"
        f"Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'.\n"
        f"<end_of_turn>\n<start_of_turn>model\n"
    )

def build_training_text(row):
    """Prompt + target completion (the gold letter) + EOS-equivalent turn close."""
    gold_raw = row.get("gold_answer", row.get("Gold_Answer", row.get("GoldAnswer")))
    if gold_raw is None:
        raise ValueError(f"No gold answer field found for row id={get_row_id(row)}")
    gold = str(gold_raw).strip().upper()
    if gold not in ("A", "B", "C", "D"):
        raise ValueError(f"Unexpected gold answer '{gold}' for row id={get_row_id(row)}")
    return prompt_chinese(row) + gold + "<end_of_turn>"

# ==========================================
# LOAD DATA
# ==========================================
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read()
    stripped = raw.strip()

    # Try whole-file JSON first (a single array or a single object,
    # possibly pretty-printed across multiple lines), but fall back to
    # line-by-line JSONL if that parse fails.
    if stripped.startswith("[") or stripped.startswith("{"):
        try:
            parsed = json.loads(stripped)
            if isinstance(parsed, list):
                return pd.DataFrame(parsed)
            if isinstance(parsed, dict):
                for key in ("data", "examples", "rows", "questions"):
                    if key in parsed and isinstance(parsed[key], list):
                        return pd.DataFrame(parsed[key])
                return pd.DataFrame([parsed])
        except json.JSONDecodeError:
            pass  # fall through to line-by-line parsing below

    rows = [json.loads(line) for line in raw.splitlines() if line.strip()]
    return pd.DataFrame(rows)

if not os.path.exists(DEV_DATA_PATH):
    raise FileNotFoundError(
        f"Chinese dev set not found at {DEV_DATA_PATH}. "
        f"Update DEV_DATA_PATH before running."
    )

df = load_jsonl(DEV_DATA_PATH)
print(f"Loaded {len(df)} Chinese dev rows. Columns: {list(df.columns)}")

# Class distribution — useful to check for imbalance before training
gold_col = "gold_answer" if "gold_answer" in df.columns else (
    "Gold_Answer" if "Gold_Answer" in df.columns else None
)
print("Gold label distribution:")
if gold_col:
    print(df[gold_col].value_counts())
else:
    print("WARNING: no gold_answer/Gold_Answer column found — check df.columns above.")

texts, skipped = [], 0
for idx, row in df.iterrows():
    try:
        texts.append(build_training_text(row))
    except Exception as e:
        skipped += 1
        print(f"WARNING: skipping row id={get_row_id(row, idx)} ({type(e).__name__}: {e})")

print(f"Built {len(texts)} training examples, skipped {skipped}")

train_dataset = Dataset.from_dict({"text": texts})

# ==========================================
# LOAD BASE MODEL + LoRA CONFIG
# ==========================================
print("Loading base model...")
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    device_map="cuda:0",
)

model = FastModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# ==========================================
# TOKENIZE
# ==========================================
def tokenize_fn(batch):
    return tokenizer(
        text=batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )

tokenized_dataset = train_dataset.map(
    tokenize_fn, batched=True, remove_columns=["text"]
)

# ==========================================
# TRAIN
# ==========================================
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=tokenized_dataset,
    args=SFTConfig(
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="indonesian_training_ckpts",
        report_to="none",
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
    ),
)

print("\n=== Starting Indonesian-only fine-tuning ===")
trainer.train()

# ==========================================
# SAVE ADAPTER
# ==========================================
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_ADAPTER_PATH)
tokenizer.save_pretrained(OUTPUT_ADAPTER_PATH)
print(f"\nSaved Chinese adapter to: {OUTPUT_ADAPTER_PATH}")
print("This path matches ADAPTER_PATHS['indonesian'] in your inference script — no changes needed there.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


/shared/envs/data-science/lib/python3.11/site-packages/unsloth/import_fixes.py:1200: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)


Loaded 366 Chinese dev rows. Columns: ['id', 'scenario_question', 'options', 'gold_answer']
Gold label distribution:
gold_answer
C       92
A       85
B       63
D       54
A, C    22
C, D    13
A, D    12
A, B    11
B, C     8
B, D     6
Name: count, dtype: int64
Built 294 training examples, skipped 72
Loading base model...
==((====))==  Unsloth 2026.7.3: Fast Gemma4 patching. Transformers: 5.14.1.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.354 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.13.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
[unsloth_zoo.log|WARNING]Unsloth: Failed to register input-embedding hook for `model.base_model.model.model.audio_tower`: `get_input_embeddings` not auto‑handled for Gemma4AudioModel; please override in the subclass.. Falling back to pre-forward hook.


...


=== Starting Indonesian-only fine-tuning ===


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 294 | Num Epochs = 3 | Total steps = 57
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 40,583,168 of 7,981,684,000 (0.51% trained)


Step,Training Loss
10,0.333638
20,0.188231
30,0.129534
40,0.147978
50,0.121955


Unsloth: Restored added_tokens_decoder metadata in indonesian_training_ckpts/checkpoint-57/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/indonesian_adapter/indonesian_adapter/tokenizer_config.json.



Saved Chinese adapter to: /mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/indonesian_adapter/indonesian_adapter
This path matches ADAPTER_PATHS['indonesian'] in your inference script — no changes needed there.


In [3]:
import json
import re
import torch
import torch.nn as nn
from tqdm import tqdm
from transformers import AutoModelForMultimodalLM, AutoProcessor
from transformers.models.gemma4 import modeling_gemma4
from peft import PeftModel

# ------------------------------------------------------------------
# Patch Gemma4ClippableLinear -> plain nn.Linear so PEFT/LoRA can
# target it. Must happen BEFORE the model is loaded (section 1 below).
# ------------------------------------------------------------------
class PatchedClippableLinear(nn.Linear):
    def __init__(self, config, in_features, out_features):
        super().__init__(in_features, out_features, bias=False)
        self.use_clipped_linears = getattr(config, "use_clipped_linears", False)
        if self.use_clipped_linears:
            self.register_buffer("input_min", torch.tensor(-float("inf")))
            self.register_buffer("input_max", torch.tensor(float("inf")))
            self.register_buffer("output_min", torch.tensor(-float("inf")))
            self.register_buffer("output_max", torch.tensor(float("inf")))

    def forward(self, x):
        if self.use_clipped_linears:
            x = torch.clamp(x, self.input_min, self.input_max)
        out = super().forward(x)
        if self.use_clipped_linears:
            out = torch.clamp(out, self.output_min, self.output_max)
        return out

modeling_gemma4.Gemma4ClippableLinear = PatchedClippableLinear

# ------------------------------------------------------------------
# 0. CONFIG — edit these paths
# ------------------------------------------------------------------
BASE_MODEL_ID = "google/gemma-4-E4B-it"
ADAPTER_DIR = "/mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/indonesian_adapter/indonesian_adapter"  # wherever you uploaded your saved adapter

TEST_SET_PATHS = {
    "si": "/mnt/nvme_storage/shared/users/amasha/PlurVa/test_set/sri_lankan_test_without_gold.jsonl",
    "zh": "/mnt/nvme_storage/shared/users/amasha/PlurVa/test_set/chinese_test_without_gold.jsonl",
    "id": "/mnt/nvme_storage/shared/users/amasha/PlurVa/test_set/indonesian_test_without_gold.jsonl",
}

RESULTS_PATHS = {
    "si": "/mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/ind-adp/results_si.json",
    "zh": "/mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/ind-adp/results_zh.json",
    "id": "/mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/ind-adp/results_id.json",
}
COMBINED_PATH = "/mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/ind-adp/predictions.jsonl"
MAX_NEW_TOKENS = 5

DATASET_NAME_MAP = {"si": "sri_lankan", "zh": "chinese", "id": "indonesian"}
SINHALA_LABEL_MAP = {"A": "A", "B": "B", "C": "Both", "D": "0"}

PROMPT_TEMPLATES = {
    "si": (
        "You are a Sinhala Expert for answering multiple-choice questions.\n"
        "    You need to always evaluate these questions based on the Sri Lankan context and provide the most accurate answer.\n"
        "    You should only respond with the correct option text, without any additional explanation or commentary.\n"
        "Question: {question}\n"
        "Option A: {A}\n"
        "Option B: {B}\n"
        "Option C: පිළිතුරු දෙකම නිවැරදියි.\n"
        "Option D: පිළිතුරු දෙකම නිවැරදි නොවේ.\n"
        "Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'."
    ),
    "zh": (
        "You are a Simplified Chinese Expert for answering multiple-choice questions.\n"
        "    You need to always evaluate these questions based on the Chinese context and provide the most accurate answer.\n"
        "    You should only respond with the correct option text, without any additional explanation or commentary.\n"
        "Question: {question}\n"
        "Option A: {A}\nOption B: {B}\nOption C: {C}\nOption D: {D}\n"
        "Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'."
    ),
    "id": (
        "You are an Indonesian Expert for answering multiple-choice questions.\n"
        "    You need to always evaluate these questions based on the Indonesian context and provide the most accurate answer.\n"
        "    You should only respond with the correct option text, without any additional explanation or commentary.\n"
        "Question: {question}\n"
        "Option A: {A}\nOption B: {B}\nOption C: {C}\nOption D: {D}\n"
        "Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'."
    ),
}

# ------------------------------------------------------------------
# 1. Load base model + LoRA adapter (once, shared across all 3 languages)
# ------------------------------------------------------------------
processor = AutoProcessor.from_pretrained(ADAPTER_DIR)  # processor saved alongside the adapter
tokenizer = processor.tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForMultimodalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype="auto",
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

# ------------------------------------------------------------------
# 2. Load test sets (raw schema: ID, Scenario (optional), Question, Option_A..D)
# ------------------------------------------------------------------
def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

def to_question_list_sinhala(raw_data):
    questions = []
    for item in raw_data:
        questions.append({
            "id": item["ID"],
            "question": item["Question"],
            "options": {
                "A": item["Option_A"],
                "B": item["Option_B"],
                # C and D are fixed literal text baked into the prompt template
            },
        })
    return questions

def to_question_list_zh_id(raw_data):
    questions = []
    for item in raw_data:
        questions.append({
            "id": item["ID"],
            "question": f"{item['Scenario']}\n\n{item['Question']}" if "Scenario" in item else item["Question"],
            "options": {
                "A": item["Option_A"],
                "B": item["Option_B"],
                "C": item["Option_C"],
                "D": item["Option_D"],
            },
        })
    return questions

raw_sinhala = load_jsonl(TEST_SET_PATHS["si"])
raw_chinese = load_jsonl(TEST_SET_PATHS["zh"])
raw_indonesian = load_jsonl(TEST_SET_PATHS["id"])

datasets = {
    "si": to_question_list_sinhala(raw_sinhala),
    "zh": to_question_list_zh_id(raw_chinese),
    "id": to_question_list_zh_id(raw_indonesian),
}
for lang, ds in datasets.items():
    print(f"Loaded {len(ds)} {lang} test questions from {TEST_SET_PATHS[lang]}")

# ------------------------------------------------------------------
# 3. Inference: build prompt, generate, parse label
# ------------------------------------------------------------------
def parse_label(text):
    match = re.search(r"\b([ABCD])\b", text.strip().upper())
    return match.group(1) if match else None

def build_prompt(lang, question, options):
    tmpl = PROMPT_TEMPLATES[lang]
    if lang == "si":
        return tmpl.format(question=question, A=options["A"], B=options["B"])
    return tmpl.format(question=question, A=options["A"], B=options["B"],
                        C=options["C"], D=options["D"])

@torch.no_grad()
def get_answer(lang, question, options, max_new_tokens=MAX_NEW_TOKENS):
    prompt = build_prompt(lang, question, options)
    messages = [{"role": "user", "content": prompt}]

    # Thinking mode is left off by default (no <|think|> token / enable_thinking
    # flag), since we only want the short answer, not a reasoning trace.
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    if lang == "zh":
        # use the fine-tuned Chinese adapter
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    else:
        # si / id: run the plain base model, adapter switched off
        with model.disable_adapter():
            output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    decoded = tokenizer.decode(output[0][input_len:], skip_special_tokens=True)
    return parse_label(decoded), decoded

def run_dataset(lang, dataset, out_path):
    results = []
    for item in tqdm(dataset, desc=lang):
        pred, raw = get_answer(lang, item["question"], item["options"])
        results.append({
            "id": item["id"],
            "lang": lang,
            "predicted_answer": pred,
            "raw_output": raw,
        })
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    return results

# ------------------------------------------------------------------
# 4. Run inference for all three languages
# ------------------------------------------------------------------
all_results = {}
for lang in ["si", "zh", "id"]:
    all_results[lang] = run_dataset(lang, datasets[lang], RESULTS_PATHS[lang])
    n_unparsed = sum(1 for r in all_results[lang] if r["predicted_answer"] is None)
    print(f"{lang}: {len(all_results[lang])} predictions -> {RESULTS_PATHS[lang]}"
          + (f"  ({n_unparsed} unparsed)" if n_unparsed else ""))

# ------------------------------------------------------------------
# 5. Convert to submission format and write combined_dataset.jsonl
# ------------------------------------------------------------------
def map_final_label(lang, pred):
    if pred is None:
        return "0"  # parsing failure -> "0"
    if lang == "si":
        return SINHALA_LABEL_MAP.get(pred, pred)
    return pred  # zh, id stay as bare A/B/C/D

combined_records = []
for lang, results in all_results.items():
    for r in results:
        combined_records.append({
            "dataset": DATASET_NAME_MAP[lang],
            "id": r["id"],
            "LLM_Output": map_final_label(lang, r["predicted_answer"]),
        })

with open(COMBINED_PATH, "w", encoding="utf-8") as f:
    for rec in combined_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"\nWrote {len(combined_records)} total submission records to {COMBINED_PATH}")

Loading weights:   0%|          | 0/2028 [00:00<?, ?it/s]

Gemma4ForConditionalGeneration LOAD REPORT from: google/gemma-4-E4B-it
Key                                                              | Status     | 
-----------------------------------------------------------------+------------+-
model.audio_tower.layers.{0...11}.self_attn.q_proj.linear.weight | UNEXPECTED | 
model.audio_tower.layers.{0...11}.self_attn.k_proj.linear.weight | UNEXPECTED | 
model.audio_tower.layers.{0...11}.self_attn.v_proj.linear.weight | UNEXPECTED | 
model.audio_tower.layers.{0...11}.self_attn.post.linear.weight   | UNEXPECTED | 
model.audio_tower.layers.{0...11}.self_attn.q_proj.weight        | MISSING    | 
model.audio_tower.layers.{0...11}.self_attn.k_proj.weight        | MISSING    | 
model.audio_tower.layers.{0...11}.self_attn.post.weight          | MISSING    | 
model.audio_tower.layers.{0...11}.self_attn.v_proj.weight        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch

Loaded 797 si test questions from /mnt/nvme_storage/shared/users/amasha/PlurVa/test_set/sri_lankan_test_without_gold.jsonl
Loaded 3210 zh test questions from /mnt/nvme_storage/shared/users/amasha/PlurVa/test_set/chinese_test_without_gold.jsonl
Loaded 1468 id test questions from /mnt/nvme_storage/shared/users/amasha/PlurVa/test_set/indonesian_test_without_gold.jsonl


si: 100%|██████████| 797/797 [01:46<00:00,  7.49it/s]


si: 797 predictions -> /mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/ind-adp/results_si.json


zh: 100%|██████████| 3210/3210 [13:29<00:00,  3.96it/s]


zh: 3210 predictions -> /mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/ind-adp/results_zh.json


id: 100%|██████████| 1468/1468 [03:01<00:00,  8.09it/s]

id: 1468 predictions -> /mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/ind-adp/results_id.json

Wrote 5475 total submission records to /mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/ind-adp/predictions.jsonl


In [1]:
import json
import re
import torch
import torch.nn as nn
from tqdm import tqdm
from transformers import AutoModelForMultimodalLM, AutoProcessor
from transformers.models.gemma4 import modeling_gemma4
from peft import PeftModel

# ------------------------------------------------------------------
# Patch Gemma4ClippableLinear -> plain nn.Linear so PEFT/LoRA can
# target it. Must happen BEFORE the model is loaded.
# ------------------------------------------------------------------
class PatchedClippableLinear(nn.Linear):
    def __init__(self, config, in_features, out_features):
        super().__init__(in_features, out_features, bias=False)
        self.use_clipped_linears = getattr(config, "use_clipped_linears", False)
        if self.use_clipped_linears:
            self.register_buffer("input_min", torch.tensor(-float("inf")))
            self.register_buffer("input_max", torch.tensor(float("inf")))
            self.register_buffer("output_min", torch.tensor(-float("inf")))
            self.register_buffer("output_max", torch.tensor(float("inf")))

    def forward(self, x):
        if self.use_clipped_linears:
            x = torch.clamp(x, self.input_min, self.input_max)
        out = super().forward(x)
        if self.use_clipped_linears:
            out = torch.clamp(out, self.output_min, self.output_max)
        return out

modeling_gemma4.Gemma4ClippableLinear = PatchedClippableLinear

# ------------------------------------------------------------------
# 0. CONFIG — Edit adapter paths & output directories here
# ------------------------------------------------------------------
BASE_MODEL_ID = "google/gemma-4-E4B-it"

# Language Adapter Paths
ADAPTER_PATHS = {
    "id": "/mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/indonesian_adapter/indonesian_adapter",
    "zh": "/mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/chinese_adapter/chinese_adapter",  # Set your Chinese adapter path
}

TEST_SET_PATHS = {
    "si": "/mnt/nvme_storage/shared/users/amasha/PlurVa/test_set/sri_lankan_test_without_gold.jsonl",
    "zh": "/mnt/nvme_storage/shared/users/amasha/PlurVa/test_set/chinese_test_without_gold.jsonl",
    "id": "/mnt/nvme_storage/shared/users/amasha/PlurVa/test_set/indonesian_test_without_gold.jsonl",
}

RESULTS_PATHS = {
    "si": "/mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/ind-chi/results_si.json",
    "zh": "/mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/ind-chi/results_zh.json",
    "id": "/mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/ind-chi/results_id.json",
}
COMBINED_PATH = "/mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/ind-chi/predictions.jsonl"
MAX_NEW_TOKENS = 5

DATASET_NAME_MAP = {"si": "sri_lankan", "zh": "chinese", "id": "indonesian"}
SINHALA_LABEL_MAP = {"A": "A", "B": "B", "C": "Both", "D": "0"}

PROMPT_TEMPLATES = {
    "si": (
        "You are a Sinhala Expert for answering multiple-choice questions.\n"
        "    You need to always evaluate these questions based on the Sri Lankan context and provide the most accurate answer.\n"
        "    You should only respond with the correct option text, without any additional explanation or commentary.\n"
        "Question: {question}\n"
        "Option A: {A}\n"
        "Option B: {B}\n"
        "Option C: පිළිතුරු දෙකම නිවැරදියි.\n"
        "Option D: පිළිතුරු දෙකම නිවැරදි නොවේ.\n"
        "Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'."
    ),
    "zh": (
        "You are a Simplified Chinese Expert for answering multiple-choice questions.\n"
        "    You need to always evaluate these questions based on the Chinese context and provide the most accurate answer.\n"
        "    You should only respond with the correct option text, without any additional explanation or commentary.\n"
        "Question: {question}\n"
        "Option A: {A}\nOption B: {B}\nOption C: {C}\nOption D: {D}\n"
        "Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'."
    ),
    "id": (
        "You are an Indonesian Expert for answering multiple-choice questions.\n"
        "    You need to always evaluate these questions based on the Indonesian context and provide the most accurate answer.\n"
        "    You should only respond with the correct option text, without any additional explanation or commentary.\n"
        "Question: {question}\n"
        "Option A: {A}\nOption B: {B}\nOption C: {C}\nOption D: {D}\n"
        "Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'."
    ),
}

# ------------------------------------------------------------------
# 1. Load Base Model and Multi-Adapter Setup
# ------------------------------------------------------------------
print("Loading processor & base model into VRAM...")
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)
tokenizer = processor.tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForMultimodalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype="auto",
    device_map="auto",
)

# Load Indonesian adapter first
print(f"Loading Indonesian adapter from: {ADAPTER_PATHS['id']}")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATHS["id"], adapter_name="id")

# Load Chinese adapter as second named adapter
print(f"Loading Chinese adapter from: {ADAPTER_PATHS['zh']}")
model.load_adapter(ADAPTER_PATHS["zh"], adapter_name="zh")

model.eval()

# ------------------------------------------------------------------
# 2. Data Preparation Helpers
# ------------------------------------------------------------------
def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

def to_question_list_sinhala(raw_data):
    questions = []
    for item in raw_data:
        questions.append({
            "id": item["ID"],
            "question": item["Question"],
            "options": {
                "A": item["Option_A"],
                "B": item["Option_B"],
            },
        })
    return questions

def to_question_list_zh_id(raw_data):
    questions = []
    for item in raw_data:
        questions.append({
            "id": item["ID"],
            "question": f"{item['Scenario']}\n\n{item['Question']}" if "Scenario" in item else item["Question"],
            "options": {
                "A": item["Option_A"],
                "B": item["Option_B"],
                "C": item["Option_C"],
                "D": item["Option_D"],
            },
        })
    return questions

raw_sinhala = load_jsonl(TEST_SET_PATHS["si"])
raw_chinese = load_jsonl(TEST_SET_PATHS["zh"])
raw_indonesian = load_jsonl(TEST_SET_PATHS["id"])

datasets = {
    "si": to_question_list_sinhala(raw_sinhala),
    "zh": to_question_list_zh_id(raw_chinese),
    "id": to_question_list_zh_id(raw_indonesian),
}

for lang, ds in datasets.items():
    print(f"Loaded {len(ds)} {lang} test questions from {TEST_SET_PATHS[lang]}")

# ------------------------------------------------------------------
# 3. Inference Engine
# ------------------------------------------------------------------
def parse_label(text):
    match = re.search(r"\b([ABCD])\b", text.strip().upper())
    return match.group(1) if match else None

def build_prompt(lang, question, options):
    tmpl = PROMPT_TEMPLATES[lang]
    if lang == "si":
        return tmpl.format(question=question, A=options["A"], B=options["B"])
    return tmpl.format(
        question=question,
        A=options["A"],
        B=options["B"],
        C=options["C"],
        D=options["D"],
    )

@torch.no_grad()
def get_answer(lang, question, options, max_new_tokens=MAX_NEW_TOKENS):
    prompt = build_prompt(lang, question, options)
    messages = [{"role": "user", "content": prompt}]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    # Explicit Routing:
    if lang == "si":
        # Sinhala -> Run plain Base Model with adapters disabled
        with model.disable_adapter():
            output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    elif lang == "zh":
        # Chinese -> Set active adapter to Chinese
        model.set_adapter("zh")
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    elif lang == "id":
        # Indonesian -> Set active adapter to Indonesian
        model.set_adapter("id")
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    decoded = tokenizer.decode(output[0][input_len:], skip_special_tokens=True)
    return parse_label(decoded), decoded

def run_dataset(lang, dataset, out_path):
    results = []
    for item in tqdm(dataset, desc=f"Processing [{lang}]"):
        pred, raw = get_answer(lang, item["question"], item["options"])
        results.append({
            "id": item["id"],
            "lang": lang,
            "predicted_answer": pred,
            "raw_output": raw,
        })
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    return results

# ------------------------------------------------------------------
# 4. Run Inference Across All Languages
# ------------------------------------------------------------------
all_results = {}
for lang in ["si", "zh", "id"]:
    all_results[lang] = run_dataset(lang, datasets[lang], RESULTS_PATHS[lang])
    n_unparsed = sum(1 for r in all_results[lang] if r["predicted_answer"] is None)
    print(
        f"Completed {lang}: {len(all_results[lang])} predictions saved to {RESULTS_PATHS[lang]}"
        + (f" ({n_unparsed} unparsed)" if n_unparsed else "")
    )

# ------------------------------------------------------------------
# 5. Format Output Predictions JSONL
# ------------------------------------------------------------------
def map_final_label(lang, pred):
    if pred is None:
        return "0"  # Parsing fallback
    if lang == "si":
        return SINHALA_LABEL_MAP.get(pred, pred)
    return pred  # zh and id maintain standard A/B/C/D

combined_records = []
for lang, results in all_results.items():
    for r in results:
        combined_records.append({
            "dataset": DATASET_NAME_MAP[lang],
            "id": r["id"],
            "LLM_Output": map_final_label(lang, r["predicted_answer"]),
        })

with open(COMBINED_PATH, "w", encoding="utf-8") as f:
    for rec in combined_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"\nSuccessfully written {len(combined_records)} predictions to {COMBINED_PATH}")

Loading processor & base model into VRAM...


tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1844 [00:00<?, ?it/s]

[transformers] Gemma4ForConditionalGeneration LOAD REPORT from: google/gemma-4-E4B-it
Key                                                                       | Status     | 
--------------------------------------------------------------------------+------------+-
model.audio_tower.layers.{0...11}.feed_forward1.ffw_layer_2.linear.weight | UNEXPECTED | 
model.audio_tower.layers.{0...11}.feed_forward2.ffw_layer_2.linear.weight | UNEXPECTED | 
model.vision_tower.encoder.layers.{0...15}.self_attn.q_proj.linear.weight | UNEXPECTED | 
model.audio_tower.layers.{0...11}.lconv1d.linear_start.linear.weight      | UNEXPECTED | 
model.vision_tower.encoder.layers.{0...15}.self_attn.o_proj.linear.weight | UNEXPECTED | 
model.audio_tower.layers.{0...11}.self_attn.q_proj.linear.weight          | UNEXPECTED | 
model.vision_tower.encoder.layers.{0...15}.mlp.down_proj.linear.weight    | UNEXPECTED | 
model.vision_tower.encoder.layers.{0...15}.mlp.gate_proj.linear.weight    | UNEXPECTED | 
model.vision_t

Loading Indonesian adapter from: /mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/indonesian_adapter/indonesian_adapter


...

Completed si: 797 predictions saved to /mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/ind-chi/results_si.json


Processing [zh]: 100%|██████████| 3210/3210 [08:30<00:00,  6.29it/s]


Completed zh: 3210 predictions saved to /mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/ind-chi/results_zh.json


Processing [id]: 100%|██████████| 1468/1468 [04:21<00:00,  5.62it/s]

Completed id: 1468 predictions saved to /mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/ind-chi/results_id.json

Successfully written 5475 predictions to /mnt/nvme_storage/shared/users/amasha/PlurVa/Gemma-4-e4b-it/ind-chi/predictions.jsonl
